# Data Availability Exploration

This notebook investigates whether Polymarket's sports-event and price-history APIs provide usable match-result markets and pre-kickoff prices for five European football leagues: La Liga, Ligue 1, Bundesliga, Serie A, and the Premier League.

The exploration starts with one known match to test how event markets and token price histories are exposed. It then broadens to league-wide event discovery, checks where league tags and series metadata differ, refines the market filter, and finishes with corrected counts of matches that have a full-time draw option. The saved outputs illustrate each investigation; running the cells may make live API requests and can produce different counts as the data changes.

Fetch the event by its slug and inspect its markets, questions, market IDs, outcomes, and CLOB token IDs. This confirms that the match is represented by separate home-win, away-win, and draw markets and identifies the token needed to query draw-price history.

In [1]:
import requests
import json

slug = "la-liga-real-madrid-vs-atletico-madrid"

response = requests.get(
    f"https://gamma-api.polymarket.com/events/slug/{slug}",
    timeout=30
)
response.raise_for_status()

event = response.json()

print("Event:", event.get("title"))
print("Number of markets:", len(event.get("markets", [])))

for market in event.get("markets", []):
    print("\nQuestion:", market.get("question"))
    print("Market ID:", market.get("id"))
    print("Outcomes:", market.get("outcomes"))
    print("Token IDs:", market.get("clobTokenIds"))

Event: La Liga: Real Madrid vs. Atletico Madrid
Number of markets: 3

Question: Will Real Madrid beat Atletico Madrid?
Market ID: 521524
Outcomes: ["Yes", "No"]
Token IDs: ["34856025776456874494676284855527473111457014817173213306913103090156657081505", "65150435851139484995645275789998864226601116639574584492318429981304658340971"]

Question: Will Atletico Madrid beat Real Madrid?
Market ID: 521525
Outcomes: ["Yes", "No"]
Token IDs: ["6638026393375045219519858591122855338585387252528948516627534363548005959898", "101031933513078295205111365494962488489841191946436002795272126271982304639298"]

Question: Will the match between Real Madrid and Atletico Madrid end in a draw?
Market ID: 521526
Outcomes: ["Yes", "No"]
Token IDs: ["6587513213372280812821769944108099146655098419206005117480086054603719740402", "59860127887670060507685922969983640527819260314107643960912146121099189047549"]


Request the token's maximum available history at hourly fidelity. The saved response is successful but contains no observations, so this first query does not establish that usable prices are unavailable; the token's history endpoint and time-window behavior need further investigation.

In [2]:
from datetime import datetime, timezone

draw_token = (
    "6587513213372280812821769944108099146655098419206005117480086054603719740402"
)

response = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": draw_token,
        "interval": "max",
        "fidelity": 60,
    },
    timeout=30,
)

print("Status:", response.status_code)
response.raise_for_status()

history = response.json().get("history", [])

print("Number of observations:", len(history))

# Show the first and last few observations
if history:
    for label, observations in [
        ("FIRST OBSERVATIONS", history[:5]),
        ("LAST OBSERVATIONS", history[-5:]),
    ]:
        print(f"\n{label}")
        for observation in observations:
            timestamp = datetime.fromtimestamp(
                observation["t"], tz=timezone.utc
            )
            print(timestamp.isoformat(), "Price:", observation["p"])
else:
    print("Response:", response.json())

Status: 200
Number of observations: 0
Response: {'history': []}


Request hourly observations from February 1 through February 10, 2025, around the scheduled February 8 match. The response now contains history, demonstrating that a bounded timestamp query retrieves data. The preview and first timestamps provide a basic check of the returned time and price fields.

In [3]:
from datetime import datetime, timezone

# February 1–10, 2025, covering the February 8 match
start = int(datetime(2025, 2, 1, tzinfo=timezone.utc).timestamp())
end = int(datetime(2025, 2, 10, tzinfo=timezone.utc).timestamp())

response = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": draw_token,
        "startTs": start,
        "endTs": end,
        "fidelity": 60,
    },
    timeout=30,
)

print("Status:", response.status_code)
print("Response preview:", response.text[:1000])
response.raise_for_status()

history = response.json().get("history", [])
print("Number of observations:", len(history))

if history:
    for observation in history[:5]:
        timestamp = datetime.fromtimestamp(
            observation["t"], tz=timezone.utc
        )
        print(timestamp.isoformat(), "Price:", observation["p"])

Status: 200
Response preview: {"history":[{"t":1738792805,"p":0.26},{"t":1738796404,"p":0.26},{"t":1738800005,"p":0.26},{"t":1738803604,"p":0.26},{"t":1738807205,"p":0.26},{"t":1738810804,"p":0.255},{"t":1738814405,"p":0.255},{"t":1738818005,"p":0.255},{"t":1738821604,"p":0.255},{"t":1738825204,"p":0.255},{"t":1738828805,"p":0.255},{"t":1738832404,"p":0.255},{"t":1738836004,"p":0.255},{"t":1738839604,"p":0.255},{"t":1738843204,"p":0.255},{"t":1738846805,"p":0.255},{"t":1738850405,"p":0.255},{"t":1738854005,"p":0.245},{"t":1738857604,"p":0.245},{"t":1738861204,"p":0.245},{"t":1738864804,"p":0.25},{"t":1738868404,"p":0.255},{"t":1738872005,"p":0.255},{"t":1738875605,"p":0.255},{"t":1738879205,"p":0.255},{"t":1738882805,"p":0.26},{"t":1738886404,"p":0.26},{"t":1738890004,"p":0.26},{"t":1738893605,"p":0.26},{"t":1738897204,"p":0.255},{"t":1738900805,"p":0.255},{"t":1738904405,"p":0.255},{"t":1738908005,"p":0.255},{"t":1738911605,"p":0.255},{"t":1738915204,"p":0.255},{"t":1738918805,"p":0.2

Convert the returned Unix timestamps to UTC, sort the observations, and set the target to one hour before scheduled kickoff. Selecting only observations at or before that target avoids using a later price. The notebook reports the selected price, its age relative to the target, and nearby observations; with hourly fidelity, the chosen value can be almost an hour old.

In [4]:
import pandas as pd

prices = pd.DataFrame(history)

prices["timestamp"] = pd.to_datetime(prices["t"], unit="s", utc=True)
prices = prices.rename(columns={"p": "draw_price"})
prices = prices.sort_values("timestamp").reset_index(drop=True)

print("First observation:", prices["timestamp"].min())
print("Last observation:", prices["timestamp"].max())

# Scheduled kickoff: February 8, 2025, 20:00 UTC
kickoff = pd.Timestamp("2025-02-08 20:00:00", tz="UTC")
target = kickoff - pd.Timedelta(hours=1)

# Only use observations at or BEFORE the target time
eligible = prices.loc[prices["timestamp"] <= target]

if eligible.empty:
    print("\nNo price available before the target.")
else:
    selected = eligible.iloc[-1]
    age_minutes = (
        target - selected["timestamp"]
    ).total_seconds() / 60

    print("\nTarget time:", target)
    print("Selected observation:", selected["timestamp"])
    print(f"Draw price: {selected['draw_price']:.1%}")
    print(f"Observation age at target: {age_minutes:.1f} minutes")

print("\nPrices around kickoff:")
print(
    prices.loc[
        prices["timestamp"].between(
            kickoff - pd.Timedelta(hours=3),
            kickoff + pd.Timedelta(hours=2)
        ),
        ["timestamp", "draw_price"]
    ].to_string(index=False)
)

First observation: 2025-02-05 22:00:05+00:00
Last observation: 2025-02-09 02:00:05+00:00

Target time: 2025-02-08 19:00:00+00:00
Selected observation: 2025-02-08 18:00:05+00:00
Draw price: 26.0%
Observation age at target: 59.9 minutes

Prices around kickoff:
                timestamp  draw_price
2025-02-08 17:00:05+00:00       0.260
2025-02-08 18:00:05+00:00       0.260
2025-02-08 19:00:05+00:00       0.260
2025-02-08 20:00:06+00:00       0.255
2025-02-08 21:00:05+00:00       0.285


Repeat the query over the two hours leading up to kickoff at one-minute fidelity, then select the latest observation no later than the same one-hour-before-kickoff target. Comparing this result with the hourly estimate shows how finer sampling can provide a fresher price; the saved sample is less than a minute from the target.

In [5]:
response = requests.get(
    "https://clob.polymarket.com/prices-history",
    params={
        "market": draw_token,
        "startTs": int(
            (kickoff - pd.Timedelta(hours=2)).timestamp()
        ),
        "endTs": int(kickoff.timestamp()),
        "fidelity": 1,
    },
    timeout=30,
)
response.raise_for_status()

fine_history = response.json().get("history", [])
print("Number of observations:", len(fine_history))

if fine_history:
    fine_prices = pd.DataFrame(fine_history)
    fine_prices["timestamp"] = pd.to_datetime(
        fine_prices["t"], unit="s", utc=True
    )
    fine_prices = fine_prices.sort_values("timestamp")

    eligible = fine_prices.loc[
        fine_prices["timestamp"] <= target
    ]

    if not eligible.empty:
        selected = eligible.iloc[-1]
        age_minutes = (
            target - selected["timestamp"]
        ).total_seconds() / 60

        print("Target:", target)
        print("Selected timestamp:", selected["timestamp"])
        print(f"Draw price: {selected['p']:.1%}")
        print(f"Age at target: {age_minutes:.2f} minutes")
    else:
        print("No observations at or before the target.")
else:
    print("Response:", response.json())

Number of observations: 120
Target: 2025-02-08 19:00:00+00:00
Selected timestamp: 2025-02-08 18:59:04+00:00
Draw price: 25.5%
Age at target: 0.93 minutes


Map the league sport codes to display names and select the corresponding entries from the `sports` metadata already available in the notebook kernel. Printing each full record exposes its league tag and series ID, which can be used to discover events. The result confirms that metadata is available for all five target leagues.

In [7]:
league_codes = {
    "epl": "Premier League",
    "lal": "La Liga",
    "fl1": "Ligue 1",
    "bun": "Bundesliga",
    "sea": "Serie A",
}

selected_leagues = [
    sport for sport in sports
    if str(sport.get("sport", "")).lower() in league_codes
]

for sport in selected_leagues:
    code = sport["sport"].lower()
    print(f"\n{league_codes[code]}")
    print(json.dumps(sport, indent=2))

print(f"\nFound {len(selected_leagues)} of 5 leagues.")


La Liga
{
  "id": 3,
  "sport": "lal",
  "name": "LaLiga",
  "image": "https://polymarket-upload.s3.us-east-2.amazonaws.com/league-icons/lal-33aa3578.png",
  "resolution": "https://www.laliga.com/en-GB",
  "ordering": "home",
  "tags": "1,780,100639,100350",
  "primaryTagId": 780,
  "series": "10193",
  "createdAt": "2025-11-05T19:27:45.399303Z"
}

Ligue 1
{
  "id": 11,
  "sport": "fl1",
  "name": "Ligue 1",
  "image": "https://polymarket-upload.s3.us-east-2.amazonaws.com/league-icons/fl1-1ddf3796.png",
  "resolution": "https://ligue1.com/en",
  "ordering": "home",
  "tags": "1,100639,102070,100350",
  "primaryTagId": 102070,
  "series": "10195",
  "createdAt": "2025-11-05T19:27:45.399303Z"
}

Bundesliga
{
  "id": 7,
  "sport": "bun",
  "name": "Bundesliga",
  "image": "https://polymarket-upload.s3.us-east-2.amazonaws.com/league-icons/bun.png",
  "resolution": "https://www.bundesliga.com/en/bundesliga",
  "ordering": "home",
  "tags": "1,1494,100639,100350",
  "primaryTagId": 1494,
  

Page through the events endpoint for each league tag, deduplicate events by ID, and look for full-match draw markets. The helper excludes halftime and other partial-match markets and recognizes draw questions or labels. The resulting counts and sample titles are a first pass, not yet a reliable league comparison: the outputs reveal both suspiciously low Serie A coverage and non-league events mixed into the Premier League results.

In [9]:
import requests
import pandas as pd
import json
import re
import time

league_tags = {
    "La Liga": 780,
    "Ligue 1": 102070,
    "Bundesliga": 1494,
    "Serie A": 100618,
    "Premier League": 306,
}

def is_full_match_draw(market):
    question = market.get("question") or ""
    label = market.get("groupItemTitle") or ""
    text = f"{question} {label}".lower()

    # Exclude halftime draws and other partial-match markets
    if re.search(
        r"half|1h|2h|first.period|second.period|extra.time",
        text
    ):
        return False

    return (
        bool(re.search(r"\bend in (?:a )?draw\b", question.lower()))
        or label.strip().lower() == "draw"
        or question.strip().lower() == "draw"
    )

session = requests.Session()
all_events = {}
match_rows = []
summary_rows = []

for league, tag_id in league_tags.items():
    print(f"\nFetching {league}...", flush=True)
    events_by_id = {}
    cursor = None
    seen_cursors = set()

    while True:
        params = {
            "tag_id": tag_id,
            "limit": 100,
            "order": "id",
            "ascending": "true",
        }

        if cursor:
            params["after_cursor"] = cursor

        response = session.get(
            "https://gamma-api.polymarket.com/events/keyset",
            params=params,
            timeout=60,
        )

        if not response.ok:
            raise RuntimeError(
                f"{league}: HTTP {response.status_code}\n"
                f"{response.text[:1000]}"
            )

        payload = response.json()
        batch = payload["events"]

        for event in batch:
            events_by_id[str(event["id"])] = event

        print(
            f"\r  Events scanned: {len(events_by_id)}",
            end="",
            flush=True,
        )

        next_cursor = payload.get("next_cursor")

        if not batch or not next_cursor:
            break

        if next_cursor in seen_cursors:
            raise RuntimeError(
                f"{league}: repeated cursor; count incomplete."
            )

        seen_cursors.add(next_cursor)
        cursor = next_cursor
        time.sleep(0.15)

    all_events[league] = list(events_by_id.values())
    league_matches = []

    for event in events_by_id.values():
        draw_markets = [
            market for market in (event.get("markets") or [])
            if is_full_match_draw(market)
        ]

        if not draw_markets:
            continue

        row = {
            "league": league,
            "event_id": str(event["id"]),
            "title": event.get("title"),
            "slug": event.get("slug"),
            "closed": event.get("closed") is True,
            "end_date": event.get("endDate"),
            "draw_market_ids": [
                market["id"] for market in draw_markets
            ],
        }
        league_matches.append(row)
        match_rows.append(row)

    closed_count = sum(row["closed"] for row in league_matches)

    summary_rows.append({
        "League": league,
        "Events scanned": len(events_by_id),
        "Matches with draw": len(league_matches),
        "Closed": closed_count,
        "Open": len(league_matches) - closed_count,
    })

    print(f"\n  Matches with draw: {len(league_matches)}")

matches_df = pd.DataFrame(match_rows)
summary_df = pd.DataFrame(summary_rows)

print("\nCOUNTS")
print(summary_df.to_string(index=False))
print("\nTOTAL MATCHES WITH DRAW:", len(matches_df))

print("\nSAMPLE TITLES")
for league in league_tags:
    print(f"\n{league}:")
    if not matches_df.empty:
        print(
            matches_df.loc[
                matches_df["league"] == league, "title"
            ].head(5).to_string(index=False)
        )


Fetching La Liga...
  Events scanned: 2247
  Matches with draw: 514

Fetching Ligue 1...
  Events scanned: 1710
  Matches with draw: 394

Fetching Bundesliga...
  Events scanned: 1485
  Matches with draw: 394

Fetching Serie A...
  Events scanned: 52
  Matches with draw: 19

Fetching Premier League...
  Events scanned: 2478
  Matches with draw: 885

COUNTS
        League  Events scanned  Matches with draw  Closed  Open
       La Liga            2247                514     493    21
       Ligue 1            1710                394     376    18
    Bundesliga            1485                394     376    18
       Serie A              52                 19      19     0
Premier League            2478                885     865    20

TOTAL MATCHES WITH DRAW: 2206

SAMPLE TITLES

La Liga:
La Liga: Real Madrid vs. Atletico Madrid
      La Liga: Real Madrid vs. Barcelona
  La Liga: Barcelona vs. Atletico Madrid
La Liga: Real Madrid vs. Atletico Madrid
     La Liga: Real Betis vs. Real Ma

Check which tag-fetched events actually carry the expected league series ID, then inspect example metadata for the apparently contaminated Premier League events. The mismatch shows that a league tag can be attached to non-league competitions, while tag-based retrieval can also miss events belonging to the intended series. This motivates using series IDs as the primary discovery key rather than trusting tag results alone.

In [10]:
league_series = {
    "La Liga": "10193",
    "Ligue 1": "10195",
    "Bundesliga": "10194",
    "Serie A": "10203",
    "Premier League": "10188",
}

for league, events in all_events.items():
    expected_series = league_series[league]

    matching = [
        event for event in events
        if any(
            str(series.get("id")) == expected_series
            for series in (event.get("series") or [])
        )
    ]

    print(f"\n{league}")
    print("Events matching league series:", len(matching))
    print("Example titles:")
    for event in matching[:8]:
        print(" -", event.get("title"))

# Inspect how the Champions League events were classified
print("\nMETADATA FOR CONTAMINATED EPL EXAMPLES")
for event in all_events["Premier League"][:5]:
    print("\nTitle:", event.get("title"))
    print("Series:", [
        (s.get("id"), s.get("title"), s.get("slug"))
        for s in (event.get("series") or [])
    ])
    print("Tags:", [
        (t.get("id"), t.get("label"))
        for t in (event.get("tags") or [])
    ])


La Liga
Events matching league series: 2178
Example titles:
 - Girona vs. Rayo Vallecano
 - Villarreal vs. Real Oviedo
 - Mallorca vs. Barcelona
 - Alaves vs. Levante UD
 - Valencia vs. Real Sociedad
 - Celta Vigo vs. Getafe
 - Athletic Club vs. Sevilla
 - Espanyol vs. Atletico Madrid

Ligue 1
Events matching league series: 1677
Example titles:
 - Stade Rennais FC 1901 vs. Olympique de Marseille
 - Racing Club de Lens vs. Olympique Lyonnais
 - AS Monaco FC vs. Le Havre AC
 - OGC Nice vs. Toulouse FC
 - Stade Brestois 29 vs. Lille OSC
 - AJ Auxerre vs. FC Lorient
 - FC Metz vs. RC Strasbourg Alsace
 - Angers SCO vs. Paris FC

Bundesliga
Events matching league series: 1442
Example titles:
 - Bayern München vs. RB Leipzig
 - 1. FC Heidenheim vs. VfL Wolfsburg
 - Eintracht Frankfurt vs. Werder Bremen
 - Bayer Leverkusen vs. 1899 Hoffenheim
 - Union Berlin vs. VfB Stuttgart
 - SC Freiburg vs. FC Augsburg
 - FC St. Pauli vs. Borussia Dortmund
 - Borussia Mönchengladbach vs. Hamburger SV

Se

Examine a newer La Liga event to see how its three result outcomes are represented as separate markets, including the draw's `groupItemTitle`. Then make a small paginated-request test for Serie A using its series ID. The examples confirm that series-based lookup returns Serie A matches even when the earlier tag scan found very few; the market fields also inform the full-match draw detector.

In [12]:
# Inspect a newer La Liga match's markets
modern_example = next(
    event for event in all_events["La Liga"]
    if any(
        str(s.get("id")) == league_series["La Liga"]
        for s in (event.get("series") or [])
    )
)

print("EXAMPLE MATCH:", modern_example.get("title"))

for market in (modern_example.get("markets") or []):
    print({
        "question": market.get("question"),
        "groupItemTitle": market.get("groupItemTitle"),
        "sportsMarketType": market.get("sportsMarketType"),
        "outcomes": market.get("outcomes"),
    })

# Check Serie A using its series ID rather than the old tag
response = session.get(
    "https://gamma-api.polymarket.com/events/keyset",
    params={
        "series_id": league_series["Serie A"],
        "limit": 5,
        "order": "id",
        "ascending": "true",
    },
    timeout=60,
)

response.raise_for_status()
serie_a_test = response.json()

print("\nSERIE A VIA SERIES:")
for event in serie_a_test.get("events", []):
    print(" -", event.get("title"))

print("More pages available:", bool(serie_a_test.get("next_cursor")))

EXAMPLE MATCH: Girona vs. Rayo Vallecano
{'question': 'Will Girona win on 2025-08-15?', 'groupItemTitle': 'Girona', 'sportsMarketType': None, 'outcomes': '["Yes", "No"]'}
{'question': 'Will Girona vs. Rayo Vallecano end in a draw?', 'groupItemTitle': 'Draw (Girona vs. Rayo Vallecano)', 'sportsMarketType': None, 'outcomes': '["Yes", "No"]'}
{'question': 'Will Rayo Vallecano win on 2025-08-15?', 'groupItemTitle': 'Rayo Vallecano', 'sportsMarketType': None, 'outcomes': '["Yes", "No"]'}

SERIE A VIA SERIES:
 - US Sassuolo Calcio vs. Napoli
 - Genoa vs. Lecce
 - AC Milan vs. US Cremonese
 - AS Roma vs. Bologna
 - Cagliari vs. Fiorentina
More pages available: True


Filter La Liga series events that the current detector did not classify as having a draw, and inspect their titles and market questions. The examples are explicitly named “More Markets” and contain spread, totals, and both-teams-to-score markets rather than the three-way match-result markets. This helps distinguish a detector miss from an event that simply does not include the draw market in its own market list.

In [13]:
missing_draw = [
    event for event in all_events["La Liga"]
    if any(
        str(s.get("id")) == league_series["La Liga"]
        for s in (event.get("series") or [])
    )
    and not any(
        is_full_match_draw(m)
        for m in (event.get("markets") or [])
    )
]

print("La Liga series events without a detected draw:", len(missing_draw))

for event in missing_draw[:10]:
    print("\nEVENT:", event.get("title"))
    print("SLUG:", event.get("slug"))
    for market in (event.get("markets") or [])[:5]:
        print("  Question:", market.get("question"))
        print("  Label:", market.get("groupItemTitle"))

La Liga series events without a detected draw: 1687

EVENT: Real Oviedo vs. Espanyol - More Markets
SLUG: lal-ovi-esp-2025-10-17-more-markets
  Question: Spread: Espanyol (-0.5)
  Label: Espanyol (-0.5)
  Question: Real Oviedo vs. Espanyol: O/U 2.5
  Label: O/U 2.5
  Question: Real Oviedo vs. Espanyol: Both Teams to Score
  Label: Both Teams to Score

EVENT: Real Sociedad vs. Sevilla - More Markets
SLUG: lal-rso-sev-2025-10-24-more-markets
  Question: Spread: Real Sociedad (-0.5)
  Label: Real Sociedad (-0.5)
  Question: Real Sociedad vs. Sevilla: O/U 2.5
  Label: O/U 2.5
  Question: Real Sociedad vs. Sevilla: Both Teams to Score
  Label: Both Teams to Score

EVENT: Valencia vs. Villarreal - More Markets
SLUG: lal-val-vil-2025-10-25-more-markets
  Question: Spread: Villarreal (-0.5)
  Label: Villarreal (-0.5)
  Question: Valencia vs. Villarreal: O/U 2.5
  Label: O/U 2.5
  Question: Valencia vs. Villarreal: Both Teams to Score
  Label: Both Teams to Score

EVENT: Athletic Club vs. Getaf

For each league, collect all pages from its series endpoint and verify each returned event's series metadata. Also retain clearly named legacy events from the earlier tag scan, since older events may lack series metadata, and deduplicate the combined set by event ID. Apply the full-match draw filter, record event and market details, and summarize matches with a draw option by closed/open status. The final table reports the corrected comparison across all five leagues; these counts reflect the saved API snapshot and can change over time.

In [14]:
# Explicit prefixes let us retain older league events without series metadata.
legacy_prefixes = {
    "La Liga": ("la liga:", "laliga:"),
    "Ligue 1": ("ligue 1:",),
    "Bundesliga": ("bundesliga:",),
    "Serie A": ("serie a:",),
    "Premier League": ("premier league:", "epl:", "english premier league:"),
}

corrected_events = {}
rows = []

for league, series_id in league_series.items():
    print(f"\nFetching {league} by series...", flush=True)

    # Retain clearly identified older league events
    events_by_id = {
        str(event["id"]): event
        for event in all_events.get(league, [])
        if (event.get("title") or "").strip().lower().startswith(
            legacy_prefixes[league]
        )
    }

    cursor = None
    seen_cursors = set()

    while True:
        params = {
            "series_id": series_id,
            "limit": 100,
            "order": "id",
            "ascending": "true",
        }
        if cursor:
            params["after_cursor"] = cursor

        response = session.get(
            "https://gamma-api.polymarket.com/events/keyset",
            params=params,
            timeout=60,
        )

        if not response.ok:
            raise RuntimeError(
                f"{league}: HTTP {response.status_code}\n"
                f"{response.text[:1000]}"
            )

        payload = response.json()
        batch = payload["events"]

        for event in batch:
            # Verify the returned event belongs to the requested series
            if any(
                str(s.get("id")) == str(series_id)
                for s in (event.get("series") or [])
            ):
                events_by_id[str(event["id"])] = event

        next_cursor = payload.get("next_cursor")
        if not batch or not next_cursor:
            break

        if next_cursor in seen_cursors:
            raise RuntimeError(f"{league}: repeated cursor.")

        seen_cursors.add(next_cursor)
        cursor = next_cursor
        time.sleep(0.15)

    corrected_events[league] = list(events_by_id.values())
    count = 0

    for event in events_by_id.values():
        draw_markets = [
            m for m in (event.get("markets") or [])
            if is_full_match_draw(m)
        ]

        if not draw_markets:
            continue

        count += 1
        rows.append({
            "league": league,
            "event_id": str(event["id"]),
            "title": event.get("title"),
            "slug": event.get("slug"),
            "closed": event.get("closed") is True,
            "end_date": event.get("endDate"),
            "draw_market_ids": [m["id"] for m in draw_markets],
        })

    print(f"  Result events with draw options: {count}")

draw_matches_df = pd.DataFrame(rows)

if draw_matches_df.empty:
    print("No draw events found.")
else:
    draw_summary = (
        draw_matches_df.groupby("league")
        .agg(
            With_draw_option=("event_id", "size"),
            Closed=("closed", "sum"),
        )
        .reindex(league_series.keys(), fill_value=0)
    )

    draw_summary["Open"] = (
        draw_summary["With_draw_option"] - draw_summary["Closed"]
    )

    print("\nCORRECTED COUNTS")
    print(draw_summary.to_string())
    print("\nTOTAL:", len(draw_matches_df))


Fetching La Liga by series...
  Result events with draw options: 514

Fetching Ligue 1 by series...
  Result events with draw options: 393

Fetching Bundesliga by series...
  Result events with draw options: 394

Fetching Serie A by series...
  Result events with draw options: 494

Fetching Premier League by series...
  Result events with draw options: 539

CORRECTED COUNTS
                With_draw_option  Closed  Open
league                                        
La Liga                      514     493    21
Ligue 1                      393     375    18
Bundesliga                   394     376    18
Serie A                      494     464    30
Premier League               539     519    20

TOTAL: 2334


Randomly sample 100 games and check whether the draw contract has a clear settled outcome. Actual final score verification needs to be done separately in the future.

In [15]:
import random
import numpy as np
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry temporary API failures
pilot_session = requests.Session()
pilot_session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    ))
)

def as_list(value):
    if isinstance(value, str):
        return json.loads(value)
    return value if isinstance(value, list) else []

def parse_time(value):
    if not value:
        return None
    timestamp = pd.to_datetime(value, utc=True, errors="coerce")
    return None if pd.isna(timestamp) else timestamp

def audit_event(league, event):
    row = {
        "league": league,
        "event_id": str(event["id"]),
        "title": event.get("title"),
        "status": None,
        "kickoff": None,
        "kickoff_source": None,
        "draw_price": np.nan,
        "price_age_minutes": np.nan,
        "settled_draw": None,
        "error": None,
    }

    draw_markets = [
        m for m in (event.get("markets") or [])
        if is_full_match_draw(m)
    ]

    if len(draw_markets) != 1:
        row["status"] = "ambiguous_draw_market"
        return row

    market = draw_markets[0]
    outcomes = as_list(market.get("outcomes"))
    tokens = as_list(market.get("clobTokenIds"))
    settlement_prices = as_list(market.get("outcomePrices"))

    yes_indexes = [
        i for i, outcome in enumerate(outcomes)
        if str(outcome).strip().lower() == "yes"
    ]

    if len(yes_indexes) != 1:
        row["status"] = "missing_yes_outcome"
        return row

    yes_index = yes_indexes[0]

    # Check settlement independently from historical-price availability.
    if market.get("closed") and len(settlement_prices) == len(outcomes):
        values = [float(p) for p in settlement_prices]
        if (
            all(p in (0.0, 1.0) for p in values)
            and sum(values) == 1.0
        ):
            row["settled_draw"] = int(values[yes_index])

    # Use explicit game-start fields only.
    # startDate/createdAt can mean market creation; endDate is not assumed kickoff.
    for source, value in [
        ("market.gameStartTime", market.get("gameStartTime")),
        ("event.startTime", event.get("startTime")),
    ]:
        kickoff_time = parse_time(value)
        if kickoff_time is not None:
            row["kickoff"] = kickoff_time
            row["kickoff_source"] = source
            break

    if row["kickoff"] is None:
        row["status"] = "missing_explicit_kickoff"
        return row

    if yes_index >= len(tokens) or not tokens[yes_index]:
        row["status"] = "missing_token"
        return row

    target_time = row["kickoff"] - pd.Timedelta(hours=1)

    response = pilot_session.get(
        "https://clob.polymarket.com/prices-history",
        params={
            "market": tokens[yes_index],
            "startTs": int(
                (target_time - pd.Timedelta(minutes=30)).timestamp()
            ),
            "endTs": int(
                (target_time + pd.Timedelta(minutes=2)).timestamp()
            ),
            "fidelity": 1,
        },
        timeout=30,
    )
    response.raise_for_status()
    observations = response.json().get("history", [])

    eligible = [
        observation for observation in observations
        if (
            target_time - pd.Timedelta(minutes=30)
        ).timestamp() <= float(observation["t"]) <= target_time.timestamp()
    ]

    if not eligible:
        row["status"] = "no_price_in_window"
        return row

    selected = max(eligible, key=lambda observation: float(observation["t"]))
    price = float(selected["p"])

    if not 0 <= price <= 1:
        row["status"] = "invalid_price"
        return row

    row["draw_price"] = price
    row["price_age_minutes"] = (
        target_time.timestamp() - float(selected["t"])
    ) / 60
    row["status"] = "price_available"
    return row


# Fixed seed makes the sample reproducible.
rng = random.Random(42)
pilot_sample = []

for league, events in corrected_events.items():
    candidates = [
        event for event in events
        if event.get("closed") is True
        and any(
            is_full_match_draw(m)
            for m in (event.get("markets") or [])
        )
    ]
    candidates = sorted(candidates, key=lambda event: str(event["id"]))

    for event in rng.sample(candidates, min(20, len(candidates))):
        pilot_sample.append((league, event))

pilot_rows = []

for index, (league, event) in enumerate(pilot_sample, start=1):
    try:
        result = audit_event(league, event)
    except Exception as exc:
        result = {
            "league": league,
            "event_id": str(event["id"]),
            "title": event.get("title"),
            "status": "request_or_parse_error",
            "error": str(exc)[:300],
        }

    pilot_rows.append(result)
    print(
        f"[{index}/{len(pilot_sample)}] {league}: "
        f"{event.get('title')} → {result['status']}",
        flush=True,
    )
    time.sleep(0.15)

pilot_df = pd.DataFrame(pilot_rows)

pilot_df["price_available"] = pilot_df["status"].eq("price_available")
pilot_df["settlement_available"] = (
    pilot_df["settled_draw"].notna()
    if "settled_draw" in pilot_df
    else False
)
pilot_df["price_and_settlement"] = (
    pilot_df["price_available"] & pilot_df["settlement_available"]
)

pilot_summary = pilot_df.groupby("league").agg(
    Sampled=("event_id", "size"),
    With_pregame_price=("price_available", "sum"),
    With_price_and_settlement=("price_and_settlement", "sum"),
)
pilot_summary["Price_coverage_pct"] = (
    100 * pilot_summary["With_pregame_price"] / pilot_summary["Sampled"]
).round(1)

print("\nPILOT SUMMARY")
print(pilot_summary.to_string())

print("\nSTATUS BREAKDOWN")
print(pd.crosstab(pilot_df["league"], pilot_df["status"]).to_string())

print("\nOVERALL")
print(
    f"Pregame price available: "
    f"{pilot_df['price_available'].sum()}/{len(pilot_df)}"
)
print(
    f"Pregame price + clear settlement: "
    f"{pilot_df['price_and_settlement'].sum()}/{len(pilot_df)}"
)

[1/100] La Liga: Sevilla vs. Barcelona → price_available
[2/100] La Liga: FC Barcelona vs. Real Oviedo → price_available
[3/100] La Liga: Villarreal CF vs. Elche CF → price_available
[4/100] La Liga: Getafe vs. Atletico Madrid → price_available
[5/100] La Liga: FC Barcelona vs. Sevilla FC → price_available
[6/100] La Liga: La Liga: Real Madrid vs. Athletic Club → ambiguous_draw_market
[7/100] La Liga: CA Osasuna vs. RCD Mallorca → price_available
[8/100] La Liga: La Liga: Real Madrid vs. Atletico Madrid → price_available
[9/100] La Liga: Real Oviedo vs. Rayo Vallecano → price_available
[10/100] La Liga: Rayo Vallecano de Madrid vs. CA Osasuna → price_available
[11/100] La Liga: Valencia vs. Villarreal → price_available
[12/100] La Liga: Levante UD vs. FC Barcelona → price_available
[13/100] La Liga: Espanyol vs. Valencia → price_available
[14/100] La Liga: Real Betis Balompié vs. Villarreal CF → price_available
[15/100] La Liga: Real Oviedo vs. Deportivo Alavés → price_available
[16/10